# Normalization-Statistics Computation

This notebook computes the per-channel normalization statistics required by
the training and inference configurations. Each stats file is a JSON dict
mapping variable names to `{min, max}` values used by the dataset classes
to scale inputs and targets into a comparable numerical range.

## What this notebook produces

- **`stats_aifs_single_all_0_24Horizons_wElev_biasCorrection.json`** — used by
  the Step-1 bias-correction config
  (`config_training_aifs_single_lt_aware_FiLM_Time_regression_STEP1.yaml`)
  and the full-pipeline inference config.
- **`stats_aifs_single_STEP2_wElevation_allData.json`** — used by the Step-2
  super-resolution configs (regression and diffusion).

The notebook also shows the pattern for computing **single-horizon stats**
(used by the per-lead specialist baselines), which can be generated by
re-running the same logic with the appropriate single-horizon dataset
files.

## Normalization scheme

We use robust **min/max scaling** based on the 0.5% and 99.5% percentiles
of each channel, computed on the training split only. This is more robust
to outliers than min/max over the raw range and more numerically stable
than mean/std for heavy-tailed distributions like precipitation. The
computation is done in PyTorch and parallelised via the standard
`DataLoader`.

## When to re-run this notebook

You need to compute new stats whenever:

- You change the **channel list** (inputs, outputs, or invariants).
- You train on a **different time range** or **different geographic domain**.
- You change the **train/validation split** (since we compute stats on the
  training portion only to avoid leakage).

If you reuse the SwAIther-Precip configs as-is on the same 2019–2023 Swiss
training subset, you do not need to re-run this notebook.

## Reproducing this notebook

Expected inputs in `../data/`:

- The preprocessed multi-horizon dataset
  (`AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc`)
- The high-res CombiPrecip target with elevation
  (`CPCH_processed_precipitation_..._wElevation.nc`)
- The single-horizon datasets if you wish to compute per-lead stats
  (`AIFS_Single_pred_concat_2019_2023_SingleHori_*.nc`)

All produced by [`notebooks/01_data_preparation.ipynb`](01_data_preparation.ipynb).


## Setup

In [ ]:
import os
import sys
import json
import torch
from torch.utils.data import DataLoader, random_split
import tqdm

# Make the swaither package importable from the notebook
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from swaither.datasets.downscaling_dataset import (
    CustomDownscalingDatasetWithLeadFast_Time,
    CustomDownscalingDatasetNoLeadFastLite_2,
)

# Output directory for stats files
STATS_OUT_DIR = "../data/stats"
os.makedirs(STATS_OUT_DIR, exist_ok=True)

## Core function

`ExtractPercentiles` takes a dataset, takes its 80/20 training split (the
same split used by the training scripts), and computes 0.5%/99.5%-percentile
min/max for each input and output channel. Results are written as JSON in
the format expected by the dataset classes.


In [ ]:
def perc(t: torch.Tensor, q: float) -> float:
    """Return the q-th percentile (0-100) of a 1D tensor using kthvalue."""
    k = int(round(q / 100.0 * t.numel()))
    k = max(0, min(k, t.numel() - 1))
    return t.view(-1).kthvalue(k + 1).values.item()


def ExtractPercentiles(
    data,
    batch_size: int,
    max_batches: int,
    lt_aware: bool,
    stat_file_name: str,
    stats_out_dir: str = STATS_OUT_DIR,
):
    """
    Compute per-channel 0.5%/99.5% percentile min/max on the training split of
    `data`, and save the result as a JSON file.

    Parameters
    ----------
    data : DownscalingDataset
        The dataset to compute statistics from (must expose `input_variables`
        and `output_variables` attributes).
    batch_size : int
        Batch size used by the internal DataLoader.
    max_batches : int
        Cap on the number of batches to sample. Together with `batch_size`,
        this controls how many examples are used to estimate percentiles.
        Increase for more stable estimates.
    lt_aware : bool
        If True, the dataset yields `(y, x, lead)` triples; if False, `(y, x)` pairs.
    stat_file_name : str
        Output filename (relative to `stats_out_dir`).
    stats_out_dir : str
        Directory to write the JSON to. Created if it doesn't exist.
    """
    # Use the same 80/20 train/val split as the training scripts, seeded for reproducibility.
    N = len(data)
    n_train = int(0.8 * N)
    n_val = N - n_train
    gen = torch.Generator().manual_seed(0)
    train_ds, _ = random_split(data, [n_train, n_val], generator=gen)

    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    # Collect a subsample to avoid OOM on large datasets.
    xs, ys = [], []
    for i, batch in tqdm.tqdm(enumerate(loader), total=max_batches):
        if i >= max_batches:
            break
        if lt_aware:
            y, x, _lead = batch
        else:
            y, x = batch
        # (B, C, H, W) -> (B, C, H*W)
        xs.append(x.flatten(2).cpu())
        ys.append(y.flatten(2).cpu())

    X = torch.cat(xs, dim=0)  # (N_samp, C_in, HW)
    Y = torch.cat(ys, dim=0)  # (N_samp, C_out, HW)

    # Per-channel robust min/max (0.5% / 99.5%)
    def channel_minmax(T: torch.Tensor):
        mins, maxs = [], []
        for c in range(T.shape[1]):
            v = T[:, c, :].contiguous().view(-1)
            lo, hi = perc(v, 0.5), perc(v, 99.5)
            if hi <= lo:
                hi = lo + 1e-6
            mins.append(lo)
            maxs.append(hi)
        return mins, maxs

    input_mins, input_maxs = channel_minmax(X)
    output_mins, output_maxs = channel_minmax(Y)

    stats = {
        "input": {
            name: {"min": float(mn), "max": float(mx)}
            for name, mn, mx in zip(data.input_variables, input_mins, input_maxs)
        },
        "output": {
            name: {"min": float(mn), "max": float(mx)}
            for name, mn, mx in zip(data.output_variables, output_mins, output_maxs)
        },
    }

    out_path = os.path.join(stats_out_dir, stat_file_name)
    with open(out_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"Saved: {out_path}")
    return stats

## 1. Step 1 (bias correction) — multi-lead with elevation

Statistics for the main multi-lead, FiLM-conditioned bias-correction model.
Input is 11 AIFS channels + 1 altitude channel; output is the coarsened
CombiPrecip on the AIFS grid.


In [ ]:
bc_dataset = CustomDownscalingDatasetWithLeadFast_Time(
    low_res_file="../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    high_res_file="../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    input_variables=[
        "total_precipitation_NoNeg",
        "convective_precipitation_NoNeg",
        "total_cloud_cover",
        "wind_10m_u",
        "wind_10m_v",
        "specific_humidity_500hPa",
        "specific_humidity_850hPa",
        "temperature_500hPa",
        "temperature_850hPa",
        "geopotential_500hPa",
        "geopotential_850hPa",
    ],
    output_variables=["precip6h_cumulative_capped"],
    invariant_variables=["altitude"],
    log_precip=True,
    crop_mult=8,
    normalize=False,  # IMPORTANT: compute stats on un-normalized data
    interp_low_res=False,  # Step 1 operates on the native low-res grid
)
print(f"Multi-lead BC dataset: {len(bc_dataset)} samples")

In [ ]:
ExtractPercentiles(
    bc_dataset,
    batch_size=256,
    max_batches=250,
    lt_aware=True,
    stat_file_name="stats_aifs_single_all_0_24Horizons_wElev_biasCorrection.json",
)

## 2. Step 2 (super-resolution) — high-res with elevation

Statistics for the CorrDiff super-resolution stage. Input is the bias-corrected
precipitation field (1 channel) + altitude; output is the high-res CombiPrecip.
The low-res precip is bilinearly interpolated onto the high-res grid before
being passed into the network (`interp_low_res=True`).


In [ ]:
sr_dataset = CustomDownscalingDatasetNoLeadFastLite_2(
    low_res_file="../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    high_res_file="../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_wElevation.nc",
    input_variables=["precip6h_cumulative_capped"],
    output_variables=["precip6h_cumulative_capped"],
    invariant_variables=["altitude"],
    log_precip=True,
    crop_mult=16,
    normalize=False,
    interp_low_res=True,
    preload_low=True,
    preload_dtype="float16",
)
print(f"Super-resolution dataset: {len(sr_dataset)} samples")

In [ ]:
ExtractPercentiles(
    sr_dataset,
    batch_size=2,
    max_batches=1800,
    lt_aware=False,
    stat_file_name="stats_aifs_single_STEP2_wElevation_allData.json",
)

## 3. Single-horizon stats (optional, for the per-lead specialist baselines)

If you intend to train the single-lead baselines (UNet Single 6h, 3day, 6day),
each needs its own stats file. The pattern is identical to Section 1 but
with the single-horizon dataset file and a different output filename. Below
is the 3-day variant as a template — adapt the paths and filenames for the
6h and 6day variants.


In [ ]:
# Template for the 3-day single-lead baseline. Comment out / change paths
# for the 6h or 6day variants.
single_lead_dataset = CustomDownscalingDatasetNoLeadFastLite_2(
    low_res_file="../data/AIFS_Single_pred_concat_2019_2023_SingleHori_3day_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    high_res_file="../data/AIFS_Single_pred_concat_2019_2023_SingleHori_3day_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    input_variables=[
        "total_precipitation_NoNeg",
        "convective_precipitation_NoNeg",
        "total_cloud_cover",
        "wind_10m_u",
        "wind_10m_v",
        "specific_humidity_500hPa",
        "specific_humidity_850hPa",
        "temperature_500hPa",
        "temperature_850hPa",
        "geopotential_500hPa",
        "geopotential_850hPa",
    ],
    output_variables=["precip6h_cumulative_capped"],
    invariant_variables=[],  # single-lead baselines do not use the altitude channel
    log_precip=True,
    crop_mult=8,
    normalize=False,
    interp_low_res=False,
    preload_low=True,
    preload_dtype="float16",
)

ExtractPercentiles(
    single_lead_dataset,
    batch_size=256,
    max_batches=250,
    lt_aware=False,
    stat_file_name="stats_aifs_single_3day_biasCorrection.json",
)

---

✅ **Done.** The generated `stats_*.json` files are now in `../data/stats/`
and ready to be referenced by the training and generation configs in `../conf/`.
